# 1. Floating point representation and roundoff error

There are several sources of errors in computation: 

- **roundoff**, a consequence of our digital representation of numbers,
- **conditioning**, a property of the problem, and
- **stability**, a property of the algorithm we use.

```{admonition} Additional resource
:class: warning
* Trefethen and Bau's **Numerical Linear Algebra**, Chapters 12-14,
* [Python documentation](https://docs.python.org/3/tutorial/floatingpoint.html#tut-fp-issues)
* [Python numerical methods textbook](https://pythonnumericalmethods.studentorg.berkeley.edu/notebooks/chapter09.02-Floating-Point-Numbers.html)
```

## 1.1 Absolute and relative error

Computers work in terms of floating-point numbers instead of reals $\mathbb{R}$. 

Let's say $\circ$ is an operator that rounds the real number $x \in \mathbb{R}$ to the nearest floating-point number $\tilde{x}$:

$$ \tilde{x} := \circ x. $$

The _absolute_ error induced by this representation is

$$ \Delta x = \circ x - x = \tilde{x} - x, $$
and the _relative_ error is

$$ \delta x = \frac{\Delta x}{x} = \frac{\tilde{x} - x}{x}. $$
We can rearrange this as

$$ \tilde{x} = (1 + \delta x)x. $$

The IEEE (Institute of Electrical and Electronics Engineers) standard _guarantees_ that 

$$ |\delta x| < \mu_M = \tfrac{1}{2}\varepsilon_M, $$
where $\mu_M$ is **machine precision** (sometimes called machine epsilon).

```{admonition} Question
Where does $\varepsilon_M$ come from, and what is it in single and double precision arithmetic?
```

## 1.2 Elements of a floating point number

Numbers in floating-point arithmetic are represented as

$$ (-1)^s (1 + f) \cdot 2^{n-b}, $$ 
where $n$ is the **exponent**, $f$ is the **mantissa**, and $s$ is the **sign**, which together define the number. The constant $b$ is predetermined (we'll see how), and is called the **bias**. Each of the three quantities $s, n, f$ are represented in base-2 by a finite number of bits (a unit that can either be 0 or 1), in the order $s$-$n$-$f$ in the machine. In double precision, they take up 1, 11, and 52 bits, respectively, whereas in single precision, they are stored in 1, 8, and 23 bits.

### 1.2.1 Sign

A single bit represents the overall sign of the number as a prefactor $(-1)^s$. Therefore $s = 0$ means the number is positive, and $s = 1$ means it's negative.

### 1.2.2 Exponent

The $n$ in the exponent is an unsigned integer, i.e. it is nonnegative. It is represented by $d$ bits. The bias $b$ is introduced so that the overall exponent can take negative values, therefore we can represent very small numbers. Since a $d$-bit integer can take values between $0$ (all bits zero) and $2^d - 1$ (all bits one), the bias is chosen as roughly half of this, so that we have roughly equal positive and negative numbers in the range of values the exponent $n-b$ can take. The bias is defined as

$$ b = 2^{d-1} - 1.$$

The cases of all exponent bits being zero or all being one are actually special cases corresponding to special numbers ($0$ and $\pm \infty$), which we'll talk about later. The size of the exponent gives an idea of the magnitude of the number.

An example, with $d = 5$ bits representing $n$, is $01001$ (in binary), where the powers of 2 are increasing right to left starting from $2^0$. Therefore,

$$ 01001 = 2^3 + 2^0 = 9. $$

### 1.2.3 Mantissa

$f$ is interpreted as the "fractional" part of the floating point number, and is responsible for precision. Due to the $+1$ in the representation, $0 \leq f < 1$. $f$ cannot equal $1$ because that would just increment the number $n$ in the exponent.

A fixed number of bits called the binary **precision**, denoted $d_f$, is used to store $f$ in base 2. Similarly to the exponent, the powers of 2 increase from right to left, the leftmost one being $2^{-1}$. Therefore, if $d_f = 5$, then the binary mantissa

$$ 01001 = 2^{-2} + 2^{-5} = \frac{9}{32}.$$

## 1.3 Special numbers

There are some values that are not naturally represented in the above system, but are nevertheless crucial for numerical computation.

Zero ($\pm 0$) is the first of these: it is defined as all **exponent bits zero**. This means the smallest-magnitude nonzero number we can have is 

$$ 0 \quad \overbrace{\underbrace{0000\ldots 1}_{d}}^{n} \quad \overbrace{\underbrace{0000\ldots 0}_{d_f}}^{f} = +(1+0) 2^{1-b},$$

which is around $2\times 10^{-308}$ in double precision.

The next special value is $\pm \infty$. It is defined as **all exponent bits one AND all mantissa bits zero**. If all exponent bits are one but the mantissa is something else, we get another special value: "Not a Number" or NaN.

The largest nonnegative number is therefore 

$$ 0 \quad \underbrace{1111\ldots 10}_{d} \quad \underbrace{11111\ldots 1}_{d_f} = +(1+1-2^{-d_f}) 2^{2^d - 2-b}.$$

## 1.4 Number spacing and machine precision

Let's revisit the mantissa. Since $f$ is represented by $d_f$ bits, it can be written as

$$ f = \sum_{i = 1}^{d_f} b_i 2^{-i}, \quad b_i \in \{0, 1\}. $$
It's useful to pull out a constant $2^{-d_f}$ to get

$$ f = 2^{-d_f}\sum_{i = 1}^{d_f} b_i 2^{d_f - i} = 2^{-d_f}z, $$
where $z$ is now an integer.

```{admonition} Question
What values can $z$ take? How many numbers are there between $2^n$ and $2^{n+1}$? What does that say about the relationship between $\varepsilon_M$ and $d$?
```

$z$ takes values in $ z \in \{0, 1, 2, \ldots, 2^{d} - 1\}$, therefore there are $2^d$ integers between two adjacent powers of $2$ in floating-point arithmetic. From this we can read off that

$$ \varepsilon_M = 2^{-d}. $$
In single precision, $d = 23$ bits are used to represent the mantissa, therefore $\varepsilon_M \approx 2\cdot 10^{-7}$ and from $\mu_M$ we expect each number to be accurate to around 7 digits. For double precision, $d = 52$ and numbers can be trusted to 16 significant digits.

Note that this doesn't mean that smallest representable number is $10^{-16}$! The relative spacing of adjacent floating point numbers is constant, meaning that there is the same number of floating point numbers between $2$ and $4$, $2^{-11}$ and $2^{-10}$, and $2^{100}$ and $2^{101}$. 

```{admonition} Exercises
:class: danger

1. Consider a (slightly bizarre) variation of floating point representation, where the exponent $n$ is represented by two bits and the fractional part of the mantissa, $f$, is represented by seven bits. What are the largest and smallest positive machine numbers that can be represented using this system (not including $\pm\infty$ or $\pm 0$)? Give your answers as both binary and decimal numbers.

2. Consider a variation of floating point representation in which the exponent is represented by $d$ bits and the fractional part of the mantissa is represented by four bits. If the largest and smallest positive machine numbers in this system are $248$ and $1.5625\times10^{-2}$, respectively, what is $d$?

3. Consider a computer that uses an arithmetic system that simply stores numbers in $n$-bit memory locations by dicing up the desired number range into $2^n-1$ even-size chunks. Explain why this is a Really Bad idea.

4. Consider yet another variation of floating point representation with
  * one sign bit ($1=$ negative),
  * five exponent bits (without sign bits, but with the appropriate bias),
  * and eight mantissa bits.
What is the decimal (base-10) value of the number that is represented as $1\ 10011\ 10001000$? Show your work.

```